In [1]:
from warnings import filterwarnings
filterwarnings(action = 'ignore')

import pandas as pd
import numpy as np
from functions import embed_optimize, calculate_padel_descriptors, calculate_rdkit_descriptors
from rdkit import Chem
import pickle


Failed to find the pandas get_adjustment() function to patch
Failed to patch pandas - PandasTools will have limited functionality


In [2]:


with open('Features_RDKit/features_to_drop_rdkit.pickle', 'rb') as inp:    
    features_to_drop = pickle.load(inp)
    features_to_drop_cationic_rdkit = features_to_drop['RDKit_cationic']
    features_to_drop_nonionic_rdkit = features_to_drop['RDKIt_non_ionic']
with open('Features_PaDEL/featured_to_drop_padel.pickle', 'rb') as inp:
    features_to_drop_padel = pickle.load(inp)
    features_to_drop_cationic_padel = features_to_drop_padel['PaDEL_cationic']
    features_to_drop_extended_padel = features_to_drop_padel['PaDEL_extended']
    features_to_drop_anionic_padel = features_to_drop_padel['PaDEL_anionic']
with open('Features_United/features_to_drop_united.pickle', 'rb') as inp:
    features_to_drop_cationic = pickle.load(inp)['United_cationic']



with open('Features_PaDEL/imputers_for_padel.pickle', 'rb') as inp:
    imputers  = pickle.load(inp)
    imputer_extended = imputers['PaDEL_extended']
    imputer_anionic = imputers['PaDEL_anionic']
    imputer_cationic = imputers['PaDEL_cationic']

#Nonionic surfactants
#with open('Features_RDKit/features_to_drop_rdkit.pickle', 'rb') as inp:    
#    features_to_drop_nonionic_rdkit = pickle.load(inp)['RDKIt_non_ionic']

**Cationic surfactants**

Leverage method

In [64]:
with open('Features_United/united_descs.pickle', 'rb') as inp:
    cationic_descs = pickle.load(inp)['United_cationic']

In [65]:
tranpX_cationic = np.transpose(cationic_descs)
xtx_cationic = np.dot(tranpX_cationic, cationic_descs)
invxtx_cationic = np.linalg.pinv(xtx_cationic)

In [66]:
p = cationic_descs.shape[1] # number of columns (variables)
n = cationic_descs.shape[0] # number of rows (samples)
    
warning_leverage_cationic = 3*(p/n)    

In [67]:
with open('Appl_domains_data/Cationic.pickle', 'wb') as out:
    pickle.dump({'invxtx':invxtx_cationic, 'warning':warning_leverage_cationic}, out)

Example

In [72]:
smi = 'COOCCCCCCCCCCCCCC[N+](C)(C)(C)'
mol = Chem.MolFromSmiles(smi)

In [73]:
descs = []
for i in range(5):
    feats_padel = calculate_padel_descriptors(smi = smi, imputer = imputer_cationic, features_to_drop = features_to_drop_cationic_padel, extended = False)
    feats_rdkit = calculate_rdkit_descriptors(smi = smi,  features_to_drop = features_to_drop_cationic_rdkit, if_non_ionic = False)
    
    feats_cationic = pd.concat([feats_padel, feats_rdkit], axis = 1)
    feats_cationic.drop(columns = features_to_drop_cationic, inplace=True)
    descs.append(feats_cationic)



In [74]:
descs = sum(descs)/len(descs)

In [75]:
leverage = np.dot(np.dot(descs, invxtx_cationic), np.transpose(descs))[0][0]
print(leverage)

35.15416872971333


**Nonionic surfactants**

In [49]:
with open('Features_RDKit/rdkit_descs.pickle', 'rb') as inp:
    nonionic_descs = pickle.load(inp)['RDKIt_non_ionic']

In [50]:
nonionic_descs = nonionic_descs.astype(float)

Calculation of AD parameters

In [51]:
tranpX_nonionic = np.transpose(nonionic_descs)
xtx_nonionic = np.dot(tranpX_nonionic, nonionic_descs)
invxtx_nonionic = np.linalg.pinv(xtx_nonionic)

In [52]:
p = nonionic_descs.shape[1] # number of columns (variables)
n = nonionic_descs.shape[0] # number of rows (samples)
    
warning_leverage_nonionic = 3*(p/n)    


In [53]:
with open('Appl_domains_data/Nonionic.pickle', 'wb') as out:
    pickle.dump({'invxtx':invxtx_nonionic, 'warning':warning_leverage_nonionic}, out)

In [54]:
smi = 'CCCCCCCCCCCCCOCCOCCOCCOCCOCCOCCOCCOCCO'
mol = Chem.MolFromSmiles(smi)

In [62]:
descs = []
for i in range(5):
    feats_nonionic = calculate_rdkit_descriptors(smi = smi,  features_to_drop = features_to_drop_nonionic_rdkit, if_non_ionic = True)
    descs.append(feats_nonionic)

leverage = np.dot(np.dot(sum(descs)/len(descs), invxtx_nonionic), np.transpose(sum(descs)/len(descs)))

In [63]:
leverage[0][0]

1.91431308940633

**Extended surfactants**

In [3]:
with open('Features_PaDEL/padel_descks.pickle', 'rb') as inp:
    extended_descs = pickle.load(inp)['PaDEL_extended']

Calculation of AD parameters

In [4]:
tranpX_extended = np.transpose(extended_descs)
xtx_extended = np.dot(tranpX_extended, extended_descs)
invxtx_extended = np.linalg.pinv(xtx_extended)

In [5]:
p = extended_descs.shape[1] # number of columns (variables)
n = extended_descs.shape[0] # number of rows (samples)
    
warning_leverage_extended = 3*(p/n)    

In [6]:
warning_leverage_extended

56.400000000000006

In [7]:
with open('Appl_domains_data/Extended.pickle', 'wb') as out:
    pickle.dump({'invxtx':invxtx_extended , 'warning':warning_leverage_extended }, out)

In [8]:
smi = 'CCCCOCCOCCOCCOC(=O)[O-]'
mol = Chem.MolFromSmiles(smi)

In [9]:
feats_extended = calculate_padel_descriptors(smi = smi, imputer = imputer_extended, features_to_drop=features_to_drop_extended_padel, extended=True)

In [10]:
leverage = np.dot(np.dot(feats_extended, invxtx_extended), np.transpose(feats_extended))[0][0]

In [11]:
print(leverage)

15.213953465716052


**Anionic surfactants**

In [114]:
with open('Features_PaDEL/padel_descks.pickle', 'rb') as inp:
    anionic_descs = pickle.load(inp)['PaDEL_anionic']

Calculation of AD parameters

In [115]:
tranpX_anionic = np.transpose(anionic_descs)
xtx_anionic = np.dot(tranpX_anionic, anionic_descs)
invxtx_anionic = np.linalg.pinv(xtx_anionic)

In [118]:
p = anionic_descs.shape[1] # number of columns (variables)
n = anionic_descs.shape[0] # number of rows (samples)
    
warning_leverage_anionic = 3*(p/n)    

In [124]:
print(warning_leverage_anionic)

48.75


In [119]:
with open('Appl_domains_data/Anionic.pickle', 'wb') as out:
    pickle.dump({'invxtx':invxtx_anionic , 'warning':warning_leverage_anionic }, out)

Example

In [131]:
smi = 'C=CCCCCCCCCCCOS(=O)(=O)[O-]'
mol = Chem.MolFromSmiles(smi)

In [132]:
feats_anionic = calculate_padel_descriptors(smi = smi, imputer = imputer_anionic, features_to_drop=features_to_drop_anionic_padel, extended = False)

In [133]:
leverage = np.dot(np.dot(feats_anionic, invxtx_anionic), np.transpose(feats_anionic))[0][0]
print(leverage)

24.955189151268165
